# Preparación de datos de pádel por año
**Borja Ibáñez Pérez · ICAI**

Coloca un Excel por año en `Datos/originales` y ejecuta las celdas en orden. Los CSV de `Datos/procesados` reúnen los años cargados y se sustituyen en cada ejecución.

## 1. Importar y definir las rutas

In [3]:
from pathlib import Path
import json
import pandas as pd

carpeta_originales = Path("Datos/originales")
salida = Path("Datos/procesados")
salida.mkdir(parents=True, exist_ok=True)

archivos = sorted(
    archivo
    for archivo in carpeta_originales.glob("*.xlsx")
    if not archivo.name.startswith("~$")
)

print("Archivos encontrados:")
for archivo in archivos:
    print("-", archivo.name)

if not archivos:
    raise FileNotFoundError(
        "No hay archivos Excel en Datos/originales."
    )

Archivos encontrados:
- datos2025.xlsx
- datos2026.xlsx


## 2. Leer las hojas originales

In [7]:
columnas_originales = [
    "Fecha", "Día de semana", "Hora de inicio", "Hora final",
    "Unidad deportiva", "Tipo de ocupación", "Uso", "Centro deportivo"
]

tablas = []

for archivo in archivos:
    with pd.ExcelFile(archivo, engine="openpyxl") as excel:
        for nombre_hoja in excel.sheet_names:

            # Excluir tu selección manual si sigue en el Excel.
            if nombre_hoja.strip().lower() == "todas las fechas padel":
                continue

            # Comprobar que la hoja tiene las columnas necesarias.
            cabecera = pd.read_excel(
                excel, sheet_name=nombre_hoja, nrows=0
            )

            if not set(columnas_originales).issubset(cabecera.columns):
                print("Hoja omitida por estructura:", archivo.name, nombre_hoja)
                continue

            tabla = pd.read_excel(
                excel,
                sheet_name=nombre_hoja,
                usecols=columnas_originales
            )

            tabla["archivo_origen"] = archivo.name
            tabla["hoja_origen"] = nombre_hoja
            tabla["fila_origen"] = range(2, len(tabla) + 2)
            tablas.append(tabla)

            print(archivo.name, "|", nombre_hoja, "->", len(tabla), "filas")

if not tablas:
    raise ValueError("No se han encontrado hojas con las columnas esperadas.")

df = pd.concat(tablas, ignore_index=True)
del tablas, tabla

print("\nTotal de registros:", len(df))

datos2025.xlsx | Ene-Abr 2025 -> 830075 filas
datos2025.xlsx | May-Ago 2025 -> 597784 filas
datos2025.xlsx | Sept-Dic 2025 -> 865073 filas
datos2026.xlsx | Ene-Abr 2026 -> 885251 filas
datos2026.xlsx | May-Ago 2026 -> 630356 filas

Total de registros: 3808539


## 3. Diagnóstico inicial


In [8]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nNulos por columna:")
print(df.isna().sum())
print("\nTipos de ocupación en el archivo completo:")
print(df["Tipo de ocupación"].value_counts(dropna=False))

Dimensiones: (3808539, 11)

Tipos de datos:
Fecha                datetime64[us]
Día de semana                   str
Hora de inicio               object
Hora final                   object
Unidad deportiva                str
Tipo de ocupación               str
Uso                             str
Centro deportivo                str
archivo_origen                  str
hoja_origen                     str
fila_origen                   int64
dtype: object

Nulos por columna:
Fecha                0
Día de semana        0
Hora de inicio       0
Hora final           0
Unidad deportiva     0
Tipo de ocupación    0
Uso                  0
Centro deportivo     0
archivo_origen       0
hoja_origen          0
fila_origen          0
dtype: int64

Tipos de ocupación en el archivo completo:
Tipo de ocupación
Reserva de Escuela                        1758462
Reserva Individual                         940933
Reserva sesión clase abierta/uso libre     913254
Reserva de Temporada                        8262

## 4. Normalizar textos para buscar
Convertimos a minúsculas, quitamos espacios exteriores y cambiamos á por a para encontrar «pádel» y «padel». Las columnas originales se mantienen intactas. Este cambio concreto basta para la palabra que buscamos; no es una normalización universal de todos los textos.

In [9]:
def normalizar_para_buscar(serie):
    return (
        serie.astype("string")
        .fillna("")
        .str.lower()
        .str.strip()
        .str.replace("á", "a", regex=False)
    )

unidad_texto = normalizar_para_buscar(df["Unidad deportiva"])
uso_texto = normalizar_para_buscar(df["Uso"])

filtro_unidad = unidad_texto.str.contains("padel", regex=False, na=False)
filtro_uso = uso_texto.str.contains("padel", regex=False, na=False)

# | significa ó: basta con que cumpla uno de los dos criterios.
filtro_padel = filtro_unidad | filtro_uso

print("Coinciden en Unidad deportiva:", int(filtro_unidad.sum()))
print("Coinciden en Uso:", int(filtro_uso.sum()))
print("Coinciden en ambas:", int((filtro_unidad & filtro_uso).sum()))
print("Solo por Uso:", int((~filtro_unidad & filtro_uso).sum()))
print("Total con el criterio O:", int(filtro_padel.sum()))

Coinciden en Unidad deportiva: 591384
Coinciden en Uso: 594981
Coinciden en ambas: 573995
Solo por Uso: 20986
Total con el criterio O: 612370


## 5. Crear la selección y revisar qué añade Uso
`loc` selecciona las filas; `copy` crea una tabla independiente.

Los que entran solo por Uso quedan marcados para revisar.

In [10]:
df_padel = df.loc[filtro_padel].copy()
df_padel["anio"] = pd.to_datetime(df_padel["Fecha"], errors="coerce", dayfirst=True).dt.year.astype("Int64")
df_padel["coincide_unidad"] = filtro_unidad.loc[filtro_padel]
df_padel["coincide_uso"] = filtro_uso.loc[filtro_padel]
df_padel["pendiente_revision_espacio"] = ~df_padel["coincide_unidad"]

solo_uso = df_padel.loc[df_padel["pendiente_revision_espacio"]].copy()

revision_uso = (
    solo_uso.groupby(
        ["anio", "Unidad deportiva", "Uso", "Tipo de ocupación"], dropna=False
    ).size().reset_index(name="registros")
    .sort_values("registros", ascending=False)
)
print("Registros adicionales por Uso:", len(solo_uso))
print(revision_uso.head(30).to_string(index=False))

Registros adicionales por Uso: 20986
 anio   Unidad deportiva                                             Uso                      Tipo de ocupación  registros
 2025 Pista 1 El Quijote                                Pádel 60 minutos                     Reserva Individual       1925
 2025   P-1 cub. cristal                                Pádel 60 minutos                     Reserva Individual       1731
 2025   P-2 cub. cristal                                Pádel 60 minutos                     Reserva Individual       1689
 2025 Pista 4 El Quijote                                Pádel 60 minutos                     Reserva Individual       1541
 2026 Pista 1 El Quijote                                Pádel 60 minutos                     Reserva Individual       1374
 2025 Pista 2 El Quijote                                Pádel 60 minutos                     Reserva Individual       1354
 2025 Pista 3 El Quijote                                Pádel 60 minutos                     Reserva I

In [11]:
carpeta_criterios = Path("Datos/criterios")
carpeta_criterios.mkdir(parents=True, exist_ok=True)
archivo_revision = carpeta_criterios / "decisiones_solo_uso.xlsx"

claves = ["anio", "Centro deportivo", "Unidad deportiva", "Uso", "Tipo de ocupación"]

revision = (
    solo_uso.groupby(claves, dropna=False)
    .size()
    .reset_index(name="registros")
    .sort_values("registros", ascending=False)
)

# Crear la plantilla una sola vez para no borrar tus decisiones al reejecutar.
if not archivo_revision.exists():
    revision["decision"] = "pendiente"
    revision["motivo"] = ""
    revision.to_excel(archivo_revision, index=False)

print("Archivo para revisar:", archivo_revision)

Archivo para revisar: Datos/criterios/decisiones_solo_uso.xlsx


In [12]:
decisiones = pd.read_excel(archivo_revision)
decisiones["decision"] = decisiones["decision"].str.strip().str.lower()

if not decisiones["decision"].isin(["mantener", "excluir", "pendiente"]).all():
    raise ValueError("Revisa la columna 'decision': hay una celda vacía o mal escrita.")

padel_revisado = df_padel.merge(
    decisiones[claves + ["decision", "motivo"]],
    on=claves,
    how="left",
    validate="many_to_one"
)

# Los casos identificados por Unidad deportiva ya estaban aceptados.
padel_revisado.loc[padel_revisado["coincide_unidad"], "decision"] = "mantener"

# Si aparece un caso nuevo de solo Uso, queda pendiente de revisar.
padel_revisado["decision"] = padel_revisado["decision"].fillna("pendiente")

print(padel_revisado["decision"].value_counts())

padel_aceptado = padel_revisado.loc[
    padel_revisado["decision"] == "mantener"
].copy()

decision
mantener    612270
excluir        100
Name: count, dtype: int64


## 6. Comprobar duplicados y separar tipos de ocupación

In [13]:
duplicados = padel_aceptado.loc[
    padel_aceptado.duplicated(subset=columnas_originales, keep=False)
].copy()

df_padel_sin_duplicados = padel_aceptado.drop_duplicates(
    subset=columnas_originales, keep="first"
).copy()

print("Filas en grupos de duplicados:", len(duplicados))
print("Repeticiones retiradas:", len(padel_aceptado) - len(df_padel_sin_duplicados))
print("\nTipos de ocupación de los registros de pádel:")
print(df_padel_sin_duplicados["Tipo de ocupación"].value_counts(dropna=False))

Filas en grupos de duplicados: 0
Repeticiones retiradas: 0

Tipos de ocupación de los registros de pádel:
Tipo de ocupación
Reserva Individual                        417661
Reserva de Escuela                        165282
Reserva de Temporada                       11628
Reserva sesión clase abierta/uso libre      6445
Reserva Interna de Temporada                6005
Reserva Interna Individual                  5247
Reserva de Club                                2
Name: count, dtype: int64


## 7. Validar fechas, horas y duración

In [15]:
def validar_padel(frame):
    work = frame.copy()
    work["fecha"] = pd.to_datetime(work["Fecha"], errors="coerce", dayfirst=True).dt.normalize()

    inicio = pd.to_timedelta(work["Hora de inicio"].astype("string"), errors="coerce")
    fin = pd.to_timedelta(work["Hora final"].astype("string"), errors="coerce")
    work["duracion_minutos"] = (fin - inicio).dt.total_seconds() / 60
    work["hora_inicio_num"] = inicio.dt.total_seconds() / 3600

    work["error_fecha"] = work["fecha"].isna()
    work["error_hora"] = (
        inicio.isna() | fin.isna()
        | (inicio < pd.Timedelta(0)) | (inicio >= pd.Timedelta(days=1))
        | (fin < pd.Timedelta(0)) | (fin > pd.Timedelta(days=1))
    )
    
    work["error_duracion"] = (
        work["duracion_minutos"].isna() | (work["duracion_minutos"] <= 0)
    )
    work["error_centro"] = work["Centro deportivo"].fillna("").astype("string").str.strip().eq("")

    columnas_error = ["error_fecha", "error_hora", "error_duracion", "error_centro"]
    ok = ~work[columnas_error].any(axis=1)
    return work.loc[ok].copy(), work.loc[~ok].copy()

padel_validos, padel_errores = validar_padel(df_padel_sin_duplicados)
print("Válidos según estas reglas:", len(padel_validos))
print("Pendientes por errores:", len(padel_errores))
print(padel_errores.head(5).to_string(index=False))

Válidos según estas reglas: 612270
Pendientes por errores: 0
Empty DataFrame
Columns: [Fecha, Día de semana, Hora de inicio, Hora final, Unidad deportiva, Tipo de ocupación, Uso, Centro deportivo, archivo_origen, hoja_origen, fila_origen, anio, coincide_unidad, coincide_uso, pendiente_revision_espacio, decision, motivo, fecha, duracion_minutos, hora_inicio_num, error_fecha, error_hora, error_duracion, error_centro]
Index: []


## 8. Obtener las reservas individuales y mantener las decisiones pendientes
Exportamos una selección amplia y otra con evidencia de pádel en la unidad deportiva. Así queda visible qué cambia si ampliamos el criterio.

Ninguna de estas tablas prueba disponibilidad, ocupación porcentual o demanda no satisfecha. Cada fila es un registro de utilización, no necesariamente una transacción independiente.

In [18]:
tipo_texto = normalizar_para_buscar(padel_validos["Tipo de ocupación"])
reservas_individuales = padel_validos.loc[tipo_texto.eq("reserva individual")].copy()
reservas_unidad_padel = reservas_individuales.loc[
    reservas_individuales["coincide_unidad"]
].copy()

print("Reservas individuales, criterio amplio:", len(reservas_individuales))
print("Con pádel identificado en la unidad:", len(reservas_unidad_padel))
print("Individuales mantenidos solo por Uso:", int((~reservas_individuales["coincide_unidad"]).sum()))
print("Centros de la selección amplia:", reservas_individuales["Centro deportivo"].nunique())
print("Primera fecha:", reservas_individuales["fecha"].min())
print("Última fecha:", reservas_individuales["fecha"].max())

resumen_centros = (
    reservas_individuales.groupby(["anio", "Centro deportivo"], as_index=False)
    .agg(registros=("fecha", "size"),
         primera_fecha=("fecha", "min"),
         ultima_fecha=("fecha", "max"),
         dias_con_registros=("fecha", "nunique"))
    .sort_values("registros", ascending=False)
)
print(resumen_centros.head(10).to_string(index=False))

Reservas individuales, criterio amplio: 417661
Con pádel identificado en la unidad: 401383
Individuales mantenidos solo por Uso: 16278
Centros de la selección amplia: 48
Primera fecha: 2025-01-02 00:00:00
Última fecha: 2026-08-31 00:00:00
 anio     Centro deportivo  registros primera_fecha ultima_fecha  dias_con_registros
 2025             Pradillo      21049    2025-01-02   2025-12-30                 320
 2025   Juan de Dios Román      14280    2025-01-02   2025-12-30                 353
 2025 Marqués de Samaranch      12640    2025-01-02   2025-12-30                 311
 2026             Pradillo      12579    2026-01-02   2026-08-31                 202
 2025               Gallur      11278    2025-01-02   2025-12-30                 327
 2025             La Elipa      10453    2025-01-02   2025-12-30                 359
 2026   Juan de Dios Román       9745    2026-01-02   2026-08-31                 241
 2025   Vicente del Bosque       9463    2025-01-02   2025-12-30                 

## 9. Exportar datos, revisión e informe de calidad

In [19]:
decisiones_por_tipo = padel_revisado["decision"].value_counts()

mantenidos = int(decisiones_por_tipo.get("mantener", 0))
excluidos = int(decisiones_por_tipo.get("excluir", 0))
pendientes = int(decisiones_por_tipo.get("pendiente", 0))

if mantenidos + excluidos + pendientes != len(df_padel):
    raise ValueError("El total de decisiones no coincide con la selección inicial.")

if mantenidos != len(padel_aceptado):
    raise ValueError("El número de registros mantenidos no coincide con padel_aceptado.")

print("Reservas individuales por año:")
print(reservas_individuales.groupby("anio").size())

df_padel.to_csv(salida / "padel_seleccion_amplia.csv", index=False, encoding="utf-8-sig")
padel_revisado.to_csv(salida / "padel_con_decisiones.csv", index=False, encoding="utf-8-sig")

padel_validos.to_csv(salida / "padel_validos.csv", index=False, encoding="utf-8-sig")
reservas_individuales.to_csv(salida / "reservas_individuales_amplias.csv", index=False, encoding="utf-8-sig")
reservas_unidad_padel.to_csv(salida / "reservas_individuales_unidad_padel.csv", index=False, encoding="utf-8-sig")
resumen_centros.to_csv(salida / "resumen_centros.csv", index=False, encoding="utf-8-sig")

revision_uso.to_csv(salida / "revision_solo_uso.csv", index=False, encoding="utf-8-sig")
padel_errores.to_csv(salida / "errores_validacion.csv", index=False, encoding="utf-8-sig")
duplicados.to_csv(salida / "duplicados_revisar.csv", index=False, encoding="utf-8-sig")

calidad = {
    "fuente": "Ayuntamiento de Madrid. Deportes. Ocupación de unidades deportivas",
    "licencia_origen": "CC BY 4.0",
    "archivos_utilizados": [archivo.name for archivo in archivos],

    "filas_originales": len(df),
    "filas_seleccion_amplia": len(df_padel),
    "filas_solo_por_uso": len(solo_uso),

    "registros_mantenidos": mantenidos,
    "registros_excluidos": excluidos,
    "registros_pendientes": pendientes,

    "repeticiones_retiradas": len(padel_aceptado) - len(df_padel_sin_duplicados),
    "validos": len(padel_validos),
    "errores": len(padel_errores),

    "individuales_criterio_amplio": len(reservas_individuales),
    "individuales_unidad_padel": len(reservas_unidad_padel),
    "individuales_mantenidos_solo_por_uso": int(
        (~reservas_individuales["coincide_unidad"]).sum()
    ),

    "nota": (
        "Solo los registros mantenidos pasan a la depuración. "
        "Las exclusiones manuales, los pendientes y los duplicados se cuentan "
        "por separado. No se imputa cero a fechas sin registros."
    )
}

(salida / "calidad_padel.json").write_text(
    json.dumps(calidad, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nInforme de calidad:")
print(json.dumps(calidad, indent=2, ensure_ascii=False))
print("\nArchivos guardados en:", salida.resolve())

Reservas individuales por año:
anio
2025    256122
2026    161539
dtype: int64

Informe de calidad:
{
  "fuente": "Ayuntamiento de Madrid. Deportes. Ocupación de unidades deportivas",
  "licencia_origen": "CC BY 4.0",
  "archivos_utilizados": [
    "datos2025.xlsx",
    "datos2026.xlsx"
  ],
  "filas_originales": 3808539,
  "filas_seleccion_amplia": 612370,
  "filas_solo_por_uso": 20986,
  "registros_mantenidos": 612270,
  "registros_excluidos": 100,
  "registros_pendientes": 0,
  "repeticiones_retiradas": 0,
  "validos": 612270,
  "errores": 0,
  "individuales_criterio_amplio": 417661,
  "individuales_unidad_padel": 401383,
  "individuales_mantenidos_solo_por_uso": 16278,
  "nota": "Solo los registros mantenidos pasan a la depuración. Las exclusiones manuales, los pendientes y los duplicados se cuentan por separado. No se imputa cero a fechas sin registros."
}

Archivos guardados en: /home/202207424/TFG analytics/Datos/procesados
